# Lecture Analyzer — Kaggle runner
Video (Hebrew/Russian, handwritten board) → Whisper transcript → keyframes → VL report → **one TaskSpec JSON per task class**.

**Before running:** Settings → Accelerator = **GPU T4**, and Internet = **On** (needed for deps + model download + Google Drive).

**Kaggle Secrets** (Add-ons → Secrets):
- `LINK_VIDEO` — the Google Drive share link to the lecture video (**required**).
- `ANTHROPIC_API_KEY` — only if you set a backend to `claude` (optional).

In [ ]:
# 1. Get/refresh the code from GitHub, then install deps.
#    Updates code in place (reset --hard) WITHOUT deleting cached artifacts under data/,
#    which is gitignored. So re-running this won't make you redo Whisper / the VL report.
#    HTTPS (Kaggle has no SSH key). Repo must be public, or use a token in the URL.
import os
# Reduce CUDA fragmentation on the T4 (must be set before torch is imported).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
REPO = 'https://github.com/cyttic/lection-analyzer.git'
if os.path.isdir('lection-analyzer/.git'):
    !git -C lection-analyzer fetch -q origin && git -C lection-analyzer reset -q --hard origin/main
else:
    !git clone -q $REPO
%cd /kaggle/working/lection-analyzer
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, 'src')
print('code + deps ready')

In [ ]:
# 2. Load Kaggle Secrets: LINK_VIDEO (Google Drive link, required) + ANTHROPIC_API_KEY (optional).
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['LINK_VIDEO'] = secrets.get_secret('LINK_VIDEO')   # Google Drive share link
print('Video link loaded')
try:
    os.environ['ANTHROPIC_API_KEY'] = secrets.get_secret('ANTHROPIC_API_KEY')
    print('Claude key loaded')
except Exception as e:
    print('No Claude key (fine if using local backend):', e)

In [ ]:
# 3. Edit config.yaml for this run (lecture id + video source from the LINK_VIDEO secret).
import os, yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['lecture'] = 'lec01'
cfg['ingest']['gdrive_url'] = os.environ['LINK_VIDEO']   # Google Drive link from the Kaggle Secret
# cfg['ingest']['source_path'] = '/kaggle/input/my-lectures/lec01.mp4'  # alt: mounted dataset
# cfg['backends']['vl_backend'] = 'claude'   # flip if local handwriting/tables are poor
yaml.safe_dump(cfg, open('config.yaml', 'w'), allow_unicode=True)
print(cfg['lecture'], cfg['ingest'], cfg['backends'])

In [ ]:
# 4a. Ingest + Whisper transcription (faster-whisper 'medium' on GPU) + keyframes.
from lection_analyzer.config import load_config
from lection_analyzer import ingest, transcribe, keyframes
cfg = load_config('config.yaml')
ingest.run(cfg)
transcript = transcribe.run(cfg)   # writes transcript.json + .srt + .txt (text w/ timecodes)
index = keyframes.run(cfg, transcript)
print('frames:', len(index.frames))

In [ ]:
# 4b. Model stages: VL report + synthesis. Builds the configured backend(s).
from lection_analyzer.backends import build_backends
from lection_analyzer import vl_report, synthesize
vl, llm = build_backends(cfg.backends)
report = vl_report.run(cfg, index, vl)
specs = synthesize.run(cfg, report, transcript, llm)
print('task classes:', [s.task_class for s in specs])

Download the `output/<lecture>/` folder (it's in `/kaggle/working`). Each `*.json` is a TaskSpec ready to register in your Agent.

**Re-run a single stage:** delete its artifact under `data/<lecture>/` (e.g. `vl_report.json`) and re-run, or use `python -m lection_analyzer.pipeline --only vl_report`.